# Lab | Music recommendations

- First re-run everything in this notebook to ensure you're comfortable with the concepts of similar audio recommendation systems based on RAG.
- Using music datasets from [this](https://github.com/Yuan-ManX/ai-audio-datasets?tab=readme-ov-file#m) github repo, create a local RAG to recommend sons based on users preferences. Example dataset from that link could be [this](https://zenodo.org/records/5794629) Artificial multitrack audio data. Feel free to find you're own datasets online, or combine the dataset used in this lab with a few you found to make some recommendations.
- Go ahead and build something great in 4 hours.

This lab demonstrate how to use Pinecone as the vector DB within an audio search application. Audio search can be used to find songs and metadata within a catalog, finding similar sounds in an audio library, or detecting who's speaking in an audio file.

We will index a set of audio recordings as vector embeddings. These vector embeddings are rich, mathematical representations of the audio recordings, making it possible to determine how similar the recordings are to one another. We will then take some new (unseen) audio recording, search through the index to find the most similar matches, and play the returned audio in this notebook.

# Install Dependencies

In [31]:
#!pip install librosa
#!pip install panns-inference

In [ ]:
# !pip install -qU pinecone-client==3.1.0 panns-inference datasets librosa

# Load Dataset

In this demo, we will use audio from the *ESC-50 dataset* — a labeled collection of 2000 environmental audio recordings, which are 5-second-long each. The dataset can be loaded from the HuggingFace model hub as follows:

In [6]:
import io
import librosa
import soundfile as sf
from datasets import load_dataset, Audio as AudioFeature

# load the dataset from huggingface model hub
data = load_dataset("ashraq/esc50", split="train")

# Newer versions of `datasets` need the extra package 'torchcodec' (plus FFmpeg) to decode audio.
# Instead, we keep the raw audio bytes (decode=False) and decode them ourselves with soundfile.
data = data.cast_column("audio", AudioFeature(decode=False))

def decode_audio(audio, sampling_rate=None):
    """Turn {'bytes': <audio file>} into {'array': mono numpy array, 'sampling_rate': ...}.
    Pass sampling_rate to resample (e.g. 32000 for PANNs)."""
    array, sr = sf.read(io.BytesIO(audio["bytes"]), dtype="float32")
    if array.ndim == 2:                      # stereo -> mono
        array = array.mean(axis=1)
    if sampling_rate and sr != sampling_rate:
        array = librosa.resample(array, orig_sr=sr, target_sr=sampling_rate)
        sr = sampling_rate
    return {"array": array, "sampling_rate": sr}

data

Repo card metadata block was not found. Setting CardData to empty.


Dataset({
    features: ['filename', 'fold', 'target', 'category', 'esc10', 'src_file', 'take', 'audio'],
    num_rows: 2000
})

The audios in the dataset are sampled at 44100Hz and loaded into NumPy arrays. Let's take a look.

In [7]:
# select the audio feature and display top three
audios = [decode_audio(a) for a in data["audio"][:3]]
audios

[{'array': array([0., 0., 0., ..., 0., 0., 0.], shape=(220500,), dtype=float32),
  'sampling_rate': 44100},
 {'array': array([-0.01184082, -0.10336304, -0.14141846, ...,  0.06985474,
          0.04049683,  0.00274658], shape=(220500,), dtype=float32),
  'sampling_rate': 44100},
 {'array': array([-0.00695801, -0.01251221, -0.01126099, ...,  0.215271  ,
         -0.00875854, -0.28903198], shape=(220500,), dtype=float32),
  'sampling_rate': 44100}]

We only need the Numpy arrays as these contain all of the audio data. We will later input these Numpy arrays directly into our embedding model to generate audio embeddings.

In [8]:
import numpy as np

# select only the audio data from the dataset and store in a numpy array
audios = np.array([decode_audio(a)["array"] for a in data["audio"]])

# Load Audio Embedding Model

We will use an audio tagging model trained from *PANNs: Large-Scale Pretrained Audio Neural Networks for Audio Pattern Recognition* paper to generate our audio embeddings. We use the *panns_inference* Python package, which provides an easy interface to load and use the model.

In [9]:
from panns_inference import AudioTagging

# load the default model into the gpu.
model = AudioTagging(checkpoint_path=None, device='mps') # change device to cpu if a gpu is not available

Checkpoint path: /Users/hamza/panns_data/Cnn14_mAP=0.431.pth
Using CPU.


## Initializing the Index

Now we need a place to store these embeddings and enable a efficient vector search through them all. To do that we use Pinecone, we can get a [free API key](https://app.pinecone.io/) and enter it below where we will initialize our connection to Pinecone and create a new index.

In [10]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
# initialize connection to pinecone (get API key at app.pinecone.io)
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY") or "YOUR_API_KEY"

In [11]:
import os
from pinecone import Pinecone

# configure client
pc = Pinecone(api_key=PINECONE_API_KEY)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [12]:
from pinecone import ServerlessSpec

cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

Create the index:

In [13]:
index_name = "audio-search-demo"

In [14]:
import time

# check if index already exists (it shouldn't if this is first time)
if index_name not in pc.list_indexes().names():
    # if does not exist, create index
    pc.create_index(
        index_name,
        dimension=2048,
        metric='cosine',
        spec=spec
    )
    # wait for index to be initialized
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)
# view index stats
index.describe_index_stats()

{'dimension': 2048,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 2000}},
 'total_vector_count': 2000}

# Generate Embeddings and Upsert

Now we generate the embeddings using the audio embedding model. We must do this in batches as processing all items at once will exhaust machine memory limits and API request limits.

In [15]:
from tqdm.auto import tqdm

# we will use batches of 64
batch_size = 64

for i in tqdm(range(0, len(audios), batch_size)):
    # find end of batch
    i_end = min(i+batch_size, len(audios))
    # extract batch
    batch = audios[i:i_end]
    # generate embeddings for all the audios in the batch
    _, emb = model.inference(batch)
    # create unique IDs
    ids = [f"{idx}" for idx in range(i, i_end)]
    # add all to upsert list
    to_upsert = list(zip(ids, emb.tolist()))
    # upsert/insert these records to pinecone
    _ = index.upsert(vectors=to_upsert)

# check that we have all vectors in index
index.describe_index_stats()

  0%|          | 0/32 [00:00<?, ?it/s]

{'dimension': 2048,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 2000}},
 'total_vector_count': 2000}

We now have *2000* audio records indexed in Pinecone, we're ready to begin querying.

# Querying

Let's first listen to an audio from our dataset. We will generate embeddings for the audio and use it to find similar audios from the Pinecone index.

In [16]:
from IPython.display import Audio, display

# we set an audio number to select from the dataset
audio_num = 400
# get the audio data of the audio number
query_audio = decode_audio(data[audio_num]["audio"])["array"]
# get the category of the audio number
category = data[audio_num]["category"]
# print the category and play the audio
print("Query Audio:", category)
Audio(query_audio, rate=44100)

Query Audio: car_horn


We have got the sound of a car horn. Let's generate an embedding for this sound.

In [17]:
# reshape query audio
query_audio = query_audio[None, :]
# get the embeddings for the audio from the model
_, xq = model.inference(query_audio)
xq.shape

(1, 2048)

We have now converted the audio into a 2048-dimension vector the same way we did for all the other audio we indexed. Let's use this to query our Pinecone index.

In [18]:
# query pinecone index with the query audio embeddings
results = index.query(vector=xq.tolist(), top_k=3)
results

{'matches': [{'id': '400', 'score': 1.00000322, 'values': []},
             {'id': '1667', 'score': 0.842213035, 'values': []},
             {'id': '1666', 'score': 0.83157444, 'values': []}],
 'namespace': '',
 'usage': {'egressBytes': 42.0, 'read_units': 1}}

Notice that the top result is the audio number 400 from our dataset, which is our query audio (the most similar item should always be the query itself). Let's listen to the top three results. 

In [19]:
# play the top 3 similar audios
for r in results["matches"]:
    # select the audio data from the databse using the id as an index
    a = decode_audio(data[int(r["id"])]["audio"])["array"]
    display(Audio(a, rate=44100))

We have great results, everything aligns with what seems to be a busy city street with car horns.

Let's write a helper function to run the queries using audio from our dataset easily. We do not need to embed these audio samples again as we have already, they are just stored in Pinecone. So, we specify the `id` of the query audio to search with and tell Pinecone to search with that.

In [20]:
def find_similar_audios(id):
    print("Query Audio:")
    # select the audio data from the databse using the id as an index
    query_audio = decode_audio(data[id]["audio"])["array"]
    # play the query audio
    display(Audio(query_audio, rate=44100))
    # query pinecone index with the query audio id
    result = index.query(id=str(id), top_k=5)
    print("Result:")
    # play the top 5 similar audios
    for r in result["matches"]:
        a = decode_audio(data[int(r["id"])]["audio"])["array"]
        display(Audio(a, rate=44100))

In [21]:
find_similar_audios(1642)

Query Audio:


Result:


Here we return a set of revving motors (they seem to either be vehicles or lawnmowers).

In [22]:
find_similar_audios(452)

Query Audio:


Result:


And now a more relaxing set of birds chirping in nature.

Let's use another audio sample from elsewhere (eg not this dataset) and see how the search performs with this.

In [120]:
#!wget https://storage.googleapis.com/audioset/miaow_16k.wav

We can load the audio into a Numpy array as follows:

In [23]:
import librosa

a, _ = librosa.load("./data/miaow_16k.wav", sr=44100)
Audio(a, rate=44100)

Now we generate the embeddings for this audio and query the Pinecone index.

In [24]:
# reshape query audio
query_audio = a[None, :]
# get the embeddings for the audio from the model
_, xq = model.inference(query_audio)

# query pinecone index with the query audio embeddings
results = index.query(vector=xq.tolist(), top_k=3)

# play the top 3 similar audios
for r in results["matches"]:
    a = decode_audio(data[int(r["id"])]["audio"])["array"]
    display(Audio(a, rate=44100))

Our audio search application has identified a set of similar cat sounds, which is excellent.

<br><hr><br>

# 🎵 My solution: Local Music Recommendation RAG

**Goal:** recommend songs based on what a user likes, using the same audio-embedding idea as the demo above.

**How it works**
1. **Dataset:** `lewtun/music_genres_small` from the Hugging Face Hub (short music clips from the Free Music Archive, labelled by genre).
2. **Embeddings:** the same **PANNs** model as the demo turns each song into a 2048-number vector. PANNs also predicts **sound tags** (e.g. *Guitar*, *Drum*, *Singing*), which we save as metadata.
3. **Local vector store:** everything stays on our machine (NumPy + pandas), no cloud vector DB needed. That is why it is a *local* RAG.
4. **Retrieval:** two ways to get recommendations:
   - **"More like these songs"**: average the vectors of songs the user likes, then find the nearest songs.
   - **"Text preferences"**: the user describes what they want ("calm acoustic guitar music"), an LLM turns this into genres and tags, and we retrieve matching songs.
5. **Generation:** an LLM writes a short explanation of *why* each song was recommended, using the retrieved metadata as context. This is the **G** in RAG.

```
User likes / text preference ──► query vector ──► local vector search ──► top songs + metadata ──► LLM explanation
```

## 1. Setup

In [26]:
# Uncomment on first run
%pip install datasets librosa panns-inference openai pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [27]:
import json
import numpy as np
import pandas as pd
from datasets import load_dataset, ClassLabel, Audio as HFAudio
from IPython.display import Audio, display
from panns_inference import labels as PANNS_LABELS
from tqdm.auto import tqdm

# PANNs (Cnn14) was trained on 32 kHz audio, so we resample our music to 32 kHz
SR = 32000
CLIP_SECONDS = 10       # we embed a 10-second clip from the middle of each song
N_SONGS = 500           # keep it small so it runs on a laptop CPU (increase if you have time)

## 2. Load the music dataset

In [28]:
music = load_dataset("lewtun/music_genres_small", split="train")
print(music)
print(music.features)

Dataset({
    features: ['audio', 'song_id', 'genre_id', 'genre'],
    num_rows: 1000
})
{'audio': Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None), 'song_id': Value('int64'), 'genre_id': Value('int64'), 'genre': Value('string')}


In [29]:
# use a random subset and resample the audio to 32 kHz
music = music.shuffle(seed=42).select(range(min(N_SONGS, len(music))))
# keep the raw bytes; decode_audio() (defined at the top) decodes + resamples to 32 kHz without torchcodec
music = music.cast_column("audio", HFAudio(decode=False))

# find the genre column (works whether genre is stored as text or as a class label)
def get_genres(ds):
    for col in ["genre", "genre_id", "label"]:
        if col in ds.column_names:
            feature = ds.features[col]
            values = ds[col]
            if isinstance(feature, ClassLabel):
                return [feature.int2str(v) for v in values]
            return [str(v) for v in values]
    return ["unknown"] * len(ds)

genres = get_genres(music)
pd.Series(genres).value_counts()

Electronic           62
Hip-Hop              56
Punk                 55
Folk                 50
Experimental         50
International        48
Chiptune / Glitch    47
Instrumental         46
Pop                  46
Rock                 40
Name: count, dtype: int64

## 3. Create embeddings + tags for every song
We reuse the PANNs `model` loaded in the demo above. `model.inference()` returns two things:
- `clipwise_output`: probabilities for 527 AudioSet sound tags → we keep the top tags as metadata
- `embedding`: the 2048-dimension vector → used for similarity search

In [30]:
def middle_clip(arr, seconds=CLIP_SECONDS, sr=SR):
    """Take a fixed-length clip from the middle of a song (pad if the song is shorter)."""
    arr = np.asarray(arr, dtype=np.float32)
    n = seconds * sr
    if len(arr) >= n:
        start = (len(arr) - n) // 2
        return arr[start:start + n]
    return np.pad(arr, (0, n - len(arr)))


# generic tags that say nothing useful about a song
IGNORE_TAGS = {"Music", "Speech", "Silence"}

def top_tags(probs, k=5):
    order = np.argsort(-probs)
    tags = [PANNS_LABELS[i] for i in order if PANNS_LABELS[i] not in IGNORE_TAGS]
    return tags[:k]


batch_size = 16
embeddings, catalog_rows = [], []

for i in tqdm(range(0, len(music), batch_size)):
    i_end = min(i + batch_size, len(music))
    batch = music.select(range(i, i_end))
    clips = np.stack([middle_clip(decode_audio(a, SR)["array"]) for a in batch["audio"]])

    clipwise, emb = model.inference(clips)

    for j in range(i_end - i):
        idx = i + j
        embeddings.append(emb[j])
        catalog_rows.append({
            "id": idx,
            "genre": genres[idx],
            "tags": top_tags(clipwise[j]),
        })

catalog = pd.DataFrame(catalog_rows).set_index("id")
catalog.head(10)

  0%|          | 0/32 [00:00<?, ?it/s]

,genre,tags
id,,
0,Chiptune / Glitch,"[Video game music, Cacophony, Soundtrack music..."
1,Instrumental,"[Electric piano, Piano, Keyboard (musical), Mu..."
2,Hip-Hop,"[Rapping, Hip hop music, Pop music, Exciting m..."
3,International,"[Rhythm and blues, Hip hop music, Pop music, R..."
4,Pop,"[Singing, Musical instrument, Grunge, Rock mus..."
5,Instrumental,"[Musical instrument, Sad music, New-age music,..."
6,Pop,"[Musical instrument, Drum, Drum kit, Singing, ..."
7,Electronic,"[New-age music, Ambient music, Musical instrum..."
8,Pop,"[Electronic music, Musical instrument, Funk, P..."


## 4. Local vector store
A tiny in-memory vector database: we normalise all vectors once, so cosine similarity becomes a simple dot product.

In [31]:
emb_matrix = np.vstack(embeddings).astype(np.float32)
emb_norm = emb_matrix / np.linalg.norm(emb_matrix, axis=1, keepdims=True)
print("Vector store shape:", emb_norm.shape)   # (number of songs, 2048)


def search(query_vec, top_k=5, exclude_ids=(), genres_filter=None):
    """Return the top_k most similar songs, with optional genre filter (like metadata filtering in Pinecone)."""
    q = query_vec / np.linalg.norm(query_vec)
    scores = emb_norm @ q
    results = []
    for idx in np.argsort(-scores):
        idx = int(idx)
        if idx in exclude_ids:
            continue
        if genres_filter and catalog.loc[idx, "genre"].lower() not in [g.lower() for g in genres_filter]:
            continue
        results.append({"id": idx, "score": float(scores[idx]), **catalog.loc[idx].to_dict()})
        if len(results) == top_k:
            break
    return results


def play(song_id):
    clip = middle_clip(decode_audio(music[song_id]["audio"], SR)["array"])
    display(Audio(clip, rate=SR))


def show_results(results, play_audio=True):
    for r in results:
        print(f"#{r['id']:>4} | score {r['score']:.3f} | {r['genre']:<14} | {', '.join(r['tags'])}")
        if play_audio:
            play(r["id"])

Vector store shape: (500, 2048)


## 5. Recommendation mode A: "More like the songs I like"
The user picks a few songs they like. We average their vectors into one **taste vector** and search for the nearest songs, excluding the ones they already know.

In [32]:
def recommend_from_liked(liked_ids, top_k=5):
    taste_vector = emb_norm[liked_ids].mean(axis=0)
    return search(taste_vector, top_k=top_k, exclude_ids=set(liked_ids))


# example: the user likes 2 songs from the same genre
example_genre = catalog["genre"].value_counts().index[0]
liked = catalog[catalog["genre"] == example_genre].index[:2].tolist()

print("User likes:")
show_results([{"id": i, "score": 1.0, **catalog.loc[i].to_dict()} for i in liked])

print("\nRecommended:")
recs = recommend_from_liked(liked)
show_results(recs)

User likes:
#   7 | score 1.000 | Electronic     | New-age music, Ambient music, Musical instrument, Synthesizer, Electronic music


#  15 | score 1.000 | Electronic     | Skateboard, Vehicle, Hip hop music, Rapping, Musical instrument



Recommended:
# 320 | score 0.871 | Electronic     | Synthesizer, Ambient music, Electronic music, Musical instrument, Keyboard (musical)


#  72 | score 0.869 | Electronic     | Electronic music, Electronica, Ambient music, Throbbing, Squish


# 247 | score 0.863 | Hip-Hop        | Soul music, Musical instrument, Funk, Rhythm and blues, Vehicle


# 124 | score 0.860 | International  | Dubstep, Electronic music, Musical instrument, Drum kit, Drum


# 462 | score 0.854 | Electronic     | Dubstep, Electronic music, Musical instrument, Sampler, Electronica


In [33]:
# quick quality check: what share of recommendations has the same genre as the liked songs?
same_genre = sum(r["genre"] == example_genre for r in recs) / len(recs)
print(f"Genre match: {same_genre:.0%}")

Genre match: 60%


## 6. Recommendation mode B: text preferences + LLM (full RAG)
1. **Understand:** an LLM converts the user's free-text preference into genres and sound tags that exist in our catalog.
2. **Retrieve:** we take songs matching those genres/tags as seeds, build a taste vector from them, and search the vector store.
3. **Generate:** the LLM explains the recommendations using only the retrieved metadata.

In [34]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

_ = load_dotenv(find_dotenv())
client = OpenAI() if os.getenv("OPENAI_API_KEY") else None
LLM_MODEL = "gpt-4o-mini"

AVAILABLE_GENRES = sorted(catalog["genre"].unique())
AVAILABLE_TAGS = sorted({t for tags in catalog["tags"] for t in tags})
print("Genres:", AVAILABLE_GENRES)
print("Number of distinct tags:", len(AVAILABLE_TAGS))

Genres: ['Chiptune / Glitch', 'Electronic', 'Experimental', 'Folk', 'Hip-Hop', 'Instrumental', 'International', 'Pop', 'Punk', 'Rock']
Number of distinct tags: 260


In [35]:
def parse_preferences(user_text):
    """Turn free text into genres + tags from our catalog (LLM, with a keyword fallback)."""
    if client is None:  # fallback without an API key: simple keyword matching
        text = user_text.lower()
        return {
            "genres": [g for g in AVAILABLE_GENRES if g.lower() in text],
            "tags": [t for t in AVAILABLE_TAGS if t.lower() in text],
        }

    prompt = f"""You map a listener's music preferences to a catalog.
Allowed genres: {AVAILABLE_GENRES}
Allowed sound tags: {AVAILABLE_TAGS}

Listener preference: "{user_text}"

Return JSON with two keys: "genres" (list of allowed genres) and "tags" (list of up to 5 allowed tags).
Only use values from the allowed lists."""
    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}],
    )
    prefs = json.loads(response.choices[0].message.content)
    # keep only values that really exist in the catalog
    prefs["genres"] = [g for g in prefs.get("genres", []) if g in AVAILABLE_GENRES]
    prefs["tags"] = [t for t in prefs.get("tags", []) if t in AVAILABLE_TAGS]
    return prefs


def recommend_from_text(user_text, top_k=5, n_seeds=5):
    prefs = parse_preferences(user_text)
    print("Understood preferences:", prefs)

    # score every song by how well its metadata matches the preferences
    def match_score(row):
        genre_hit = 2 if row["genre"] in prefs["genres"] else 0
        tag_hits = len(set(row["tags"]) & set(prefs["tags"]))
        return genre_hit + tag_hits

    scores = catalog.apply(match_score, axis=1)
    seeds = scores[scores > 0].sort_values(ascending=False).index[:n_seeds].tolist()

    if not seeds:
        print("No matching songs found for these preferences.")
        return prefs, []

    taste_vector = emb_norm[seeds].mean(axis=0)
    results = search(taste_vector, top_k=top_k, genres_filter=prefs["genres"] or None)
    return prefs, results


def explain(user_text, results):
    """Generation step: the LLM explains the picks using only the retrieved metadata."""
    if client is None or not results:
        return "(No LLM explanation: missing OPENAI_API_KEY or no results.)"
    context = "\n".join(
        f"- Song #{r['id']}: genre={r['genre']}, sound tags={', '.join(r['tags'])}, similarity={r['score']:.2f}"
        for r in results
    )
    prompt = f"""A listener asked for: "{user_text}"

Here are the songs retrieved from our catalog:
{context}

In 3-5 short sentences, explain why these songs fit the request.
Only use the information above. If a song does not fit well, say so honestly."""
    response = client.chat.completions.create(
        model=LLM_MODEL, temperature=0.3,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

In [36]:
query = "I want calm music with acoustic guitar"
prefs, results = recommend_from_text(query)
show_results(results)
print("\n🤖", explain(query, results))

Understood preferences: {'genres': ['Instrumental', 'Folk', 'Electronic'], 'tags': ['Acoustic guitar', 'Ambient music', 'Background music', 'Tender music']}
# 292 | score 0.940 | Instrumental   | Sad music, New-age music, Background music, Tender music, Musical instrument


# 439 | score 0.930 | Instrumental   | Musical instrument, Guitar, Plucked string instrument, Tender music, Background music


#   5 | score 0.929 | Instrumental   | Musical instrument, Sad music, New-age music, Background music, Theme music


# 431 | score 0.910 | Instrumental   | Background music, Musical instrument, Sad music, Electronic music, Soundtrack music


# 492 | score 0.910 | Instrumental   | Background music, Theme music, New-age music, Sad music, Musical instrument



🤖 The songs retrieved from the catalog all fall under the instrumental genre and feature acoustic guitar or similar musical elements, making them suitable for a listener seeking calm music. Songs #439 and #292 specifically highlight the acoustic guitar and tender music tags, aligning closely with the request for calmness. Additionally, songs #5, #431, and #492 incorporate elements of sad and new-age music, which can also contribute to a serene listening experience. Overall, these selections provide a soothing backdrop, perfect for relaxation.


In [37]:
query = "Give me energetic music with drums and electric guitar"
prefs, results = recommend_from_text(query)
show_results(results)
print("\n🤖", explain(query, results))

Understood preferences: {'genres': ['Electronic', 'Rock', 'Punk'], 'tags': ['Drum', 'Drum kit', 'Electric guitar', 'Exciting music', 'Dance music']}
#  40 | score 0.931 | Punk           | Musical instrument, Guitar, Heavy metal, Drum kit, Drum


# 126 | score 0.930 | Punk           | Rock music, Psychedelic rock, Grunge, Punk rock, Drum


# 388 | score 0.929 | Punk           | Drum kit, Drum, Bass drum, Musical instrument, Snare drum


# 475 | score 0.921 | Rock           | Drum kit, Drum, Bass drum, Musical instrument, Snare drum


# 205 | score 0.914 | Rock           | Psychedelic rock, Drum, Drum kit, Rock music, Bass drum



🤖 The songs retrieved from the catalog fit the request for energetic music with drums and electric guitar, primarily from the punk and rock genres. Songs #40, #126, and #388 all feature prominent drum elements and guitar sounds, making them suitable for an energetic vibe. Additionally, Songs #475 and #205 also include strong drum components and align with the rock genre, further enhancing their energetic qualities. Overall, these selections emphasize the requested instrumentation and genre characteristics effectively.


In [38]:
query = "Something with singing, maybe hip-hop or pop"
prefs, results = recommend_from_text(query)
show_results(results)
print("\n🤖", explain(query, results))

Understood preferences: {'genres': ['Hip-Hop', 'Pop'], 'tags': ['Singing', 'Female singing', 'Male speech, man speaking', 'Vocal music', 'Happy music']}
# 408 | score 0.903 | Hip-Hop        | Rhythm and blues, Singing, Independent music, Hip hop music, Rapping


# 264 | score 0.893 | Hip-Hop        | Hip hop music, Rapping, Musical instrument, Drum kit, Drum


#   2 | score 0.888 | Hip-Hop        | Rapping, Hip hop music, Pop music, Exciting music, Punk rock


# 171 | score 0.885 | Hip-Hop        | Rapping, Hip hop music, Reggae, Singing, Funk


# 160 | score 0.885 | Hip-Hop        | Rapping, Hip hop music, Reggae, Singing, Exciting music



🤖 The songs retrieved from the catalog fit the listener's request for something with singing in the hip-hop or pop genre. Song #408 prominently features singing alongside hip-hop elements, making it a strong match. Song #171 and Song #160 also include singing and blend hip-hop with reggae influences, enhancing their appeal. While Song #264 and Song #2 are more focused on rapping and musical instruments, they still maintain a hip-hop essence but may not emphasize singing as much as the others. Overall, Songs #408, #171, and #160 are the best fits for the request.


## 7. Bonus: recommend from your own audio file
Any audio file works: it is embedded the same way and searched against the catalog.

In [39]:
import librosa

def recommend_from_file(path, top_k=5):
    audio, _ = librosa.load(path, sr=SR, mono=True)
    _, emb = model.inference(middle_clip(audio)[None, :])
    return search(emb[0], top_k=top_k)

# Example (put any .mp3/.wav file in ./data first):
# show_results(recommend_from_file("./data/my_song.mp3"))

## 8. Reflection

*(Check these points against your own outputs and adjust them if your run behaved differently.)*

- **What worked:** "More like these songs" recommendations mostly stayed in the same genre, which shows that PANNs embeddings capture musical style even though the model was trained on general sounds, not specifically on music genres.
- **Tags as metadata:** PANNs sound tags (*Guitar*, *Drum*, *Singing*...) let us connect **text** preferences to **audio** vectors without a text-audio model.
- **Role of the LLM:** the LLM is only used to *understand* the request and to *explain* the results. The actual recommendation comes from vector similarity, which keeps the answers grounded in the catalog.
- **Limitations:**
  - PANNs was not trained for music similarity, so genres with similar instruments (e.g. rock vs. metal) can get mixed up.
  - We only embed a 10-second clip per song, which may miss intros or changes in the song.
  - Small subset (500 songs) and only genre + tags as metadata (no artist or title).
- **Possible improvements:** a music-specific embedding model (e.g. CLAP for text-to-audio search), more metadata (artist, tempo, mood), and a persistent vector DB such as Pinecone or Chroma for a larger catalog.

# Delete the Index

Delete the index once you are sure that you do not want to use it anymore. Once the index is deleted, you cannot use it again.

In [127]:
pc.delete_index(index_name)